# 01 経済成長モデル

**G1：国家・世界経済の成長と資源配分の最適化**（補遺トピック）

- 架空の国家の**状態変数**（人的資本・技術・インフラ・健康）と、そこから計算する **GDP・生活水準**
- 投資 → 状態変数の更新（**収穫逓減・遅延・減耗**）→ GDP・生活水準 を 1 年ずつくり返す（**時系列シミュレーション**）
- 計算：`economic_model.py`・`simulation.py`

> **補遺トピック G1・教育用の架空モデル**です。国名・経済指標・投資効果の係数はすべて架空で、現実の国家の GDP・経済成長率・政策効果を予測するものではありません。実際の政策判断に使わないでください。

## 準備：実験環境（最初に必ず実行）

G1 リポジトリの `.py`（計算の本体）を import できるように、作業ディレクトリ（G1 のルート）を自動で探して設定します。

| 実行場所 | 準備 |
| --- | --- |
| ローカルの Jupyter / VSCode | `notebooks/` の中で開けば、そのまま動きます |
| Colab（方法A：GitHub から clone） | 下のセルの `REPO_URL` に**自分たちのリポジトリの URL** を入れる |
| Colab（方法B：ZIP をアップロード） | G1 フォルダを ZIP にして `/content` にアップロードしておく（またはセルの実行中にアップロード） |

Colab の CPU ランタイムで動きます（GPU・TPU は不要）。`textbook/`（教科書のコード）は公開リポジトリに含めない約束です。`textbook/` がないと import が失敗します。

In [ ]:
#@title 準備（G1 の .py を読み込む）
import os, sys, glob, zipfile, subprocess
from pathlib import Path

# ---- 方法A：GitHub から clone する場合だけ設定する（URL は自分たちのものを入れる）----
REPO_URL = ""                                     # 例："https://github.com/<team>/<repo>.git"（空なら clone しない）
REPO_SUBDIR = "g1-economic-growth-optimization"   # リポジトリの中の G1 フォルダ（ルートが G1 なら ""）

IN_COLAB = "google.colab" in sys.modules


def is_g1_root(p):
    return (Path(p) / "economic_model.py").exists() and (Path(p) / "optimization.py").exists()


def find_g1_root():
    for p in [Path.cwd(), Path.cwd().parent]:          # ローカル：notebooks/ の中、または G1 のルート
        if is_g1_root(p):
            return p.resolve()
    if not IN_COLAB:
        return None
    if REPO_URL:                                        # Colab 方法A
        dest = Path("/content/repo")
        if not dest.exists():
            subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(dest)], check=True)
        cand = dest / REPO_SUBDIR if REPO_SUBDIR else dest
        if is_g1_root(cand):
            return cand
    zips = glob.glob("/content/*.zip")                  # Colab 方法B
    if not zips:
        from google.colab import files
        print("G1 フォルダの ZIP を選んでアップロードしてください")
        zips = ["/content/" + name for name in files.upload()]
    for z in zips:
        with zipfile.ZipFile(z) as zf:
            zf.extractall("/content/g1_upload")
    for hit in glob.glob("/content/g1_upload/**/economic_model.py", recursive=True):
        if is_g1_root(Path(hit).parent):
            return Path(hit).parent
    return None


G1_ROOT = find_g1_root()
if G1_ROOT is None:
    raise RuntimeError("G1 のフォルダが見つかりません。上の説明（方法A／方法B）を確認してください。")
os.chdir(G1_ROOT)
if str(G1_ROOT) not in sys.path:
    sys.path.insert(0, str(G1_ROOT))
if not (G1_ROOT / "textbook" / "fixed_queue.py").exists():
    print("⚠️ textbook/ がありません。economic_model.py・optimization.py の import が失敗します。")

# ---- 既存の .py（計算の本体）と、グラフ ----
import matplotlib.pyplot as plt
import economic_model as em
import simulation as sim
import allocation_strategies as al
import optimization as op
import evaluation as ev
import growth_charts as gc

gc.setup_style()                        # 文字を大きくするなら gc.setup_style(1.3)
PARAMS = em.load_economy()              # 架空の国家のパラメータ
SCENARIOS = al.load_scenarios()         # 比較用の配分シナリオ
LABELS = {sc["id"]: sc["label"] for sc in SCENARIOS}
YEARS = PARAMS["simulation_years"]
STEP = 0.1                              # 最適化の配分の刻み（10%）
M = al.units_count(STEP)

SAVE_FIGURES = False                    # True にすると、グラフを notebooks/figures/ に PNG で保存する
FIG_DIR = "notebooks/figures"


def show(fig, name):
    """グラフを表示し、SAVE_FIGURES = True なら PNG も保存する"""
    if SAVE_FIGURES:
        print("saved:", gc.save(fig, f"{FIG_DIR}/{name}.png"))
    plt.show()


def table(rows, columns):
    """辞書のリストを表で表示する。columns = [(キー, 見出し, 書式)]。pandas があれば DataFrame で表示"""
    try:
        import pandas as pd
        from IPython.display import display
        display(pd.DataFrame([{h: ("-" if r.get(k) is None else f.format(r[k])) for k, h, f in columns} for r in rows]))
    except ImportError:
        print(ev.format_table(rows, columns))


def alloc_cols():
    return [(s, em.SECTOR_LABELS[s], "{:.0%}") for s in em.SECTORS]


print("G1_ROOT:", G1_ROOT)
print(PARAMS["country"], "｜", YEARS, "年、年間予算", PARAMS["annual_investment_budget"], "（架空の単位）")
print("シナリオ:", [sc["label"] for sc in SCENARIOS])

## 1. 状態変数とパラメータ

| 分野 | 投資が改善する状態変数 | 効率 | 遅延 | 減耗率 |
| --- | --- | --- | --- | --- |
| 教育 | 人的資本 Human Capital | `education_efficiency` | `education_delay` | `human_capital_depreciation` |
| 研究開発 | 技術 Technology | `rd_efficiency` | `rd_delay` | `technology_depreciation` |
| インフラ | インフラ Infrastructure | `infrastructure_efficiency` | `infrastructure_delay` | `infrastructure_depreciation` |
| 医療 | 健康 Health | `healthcare_efficiency` | `healthcare_delay` | `health_depreciation` |

状態変数はすべて「初期値 = 1 の指数」です。値は**架空**です。

In [ ]:
rows = []
for sec in em.SECTORS:
    st = em.STATE_OF_SECTOR[sec]
    rows.append({"sector": em.SECTOR_LABELS[sec], "state": st, "efficiency": PARAMS[em.EFFICIENCY_KEY[sec]],
                 "delay": PARAMS[em.DELAY_KEY[sec]], "depreciation": PARAMS[em.DEPRECIATION_KEY[st]],
                 "gdp_elasticity": PARAMS[em.ELASTICITY_KEY[st]]})
table(rows, [("sector", "sector", "{}"), ("state", "state", "{}"), ("efficiency", "efficiency", "{}"),
             ("delay", "delay (years)", "{}"), ("depreciation", "depreciation", "{}"),
             ("gdp_elasticity", "GDP elasticity", "{}")])

## 2. 状態変数の更新

$$ \text{効果}_k(t) = \text{効率}_k \times \text{投資}_k(t)^{\,r} \qquad (r = 0.5 \text{ なら平方根：収穫逓減}) $$

$$ X_k(t) = (1 - \delta_k)\, X_k(t-1) + \text{効果}_k(t - d_k) \qquad (\delta_k：減耗率、d_k：遅延) $$

遅延 $d_k$ 年の効果は、教科書の**キュー（FixedQueue）**に入れておき、$d_k$ 年後に取り出します。

In [ ]:
for inv in [0, 25, 50, 100]:
    print(f"投資 {inv:3}: 研究開発の効果 {em.investment_effect(inv, PARAMS['rd_efficiency'], PARAMS['returns_exponent']):.4f}"
          f"、次の 10 単位の追加効果 {em.marginal_effect(inv, PARAMS['rd_efficiency'], PARAMS['returns_exponent'], 10):.4f}")
line = em.DelayLine(3)
print("遅延 3 年のキュー：入れた効果 [1, 2, 3, 4, 5] → 届いた効果", [line.push(v) for v in [1.0, 2.0, 3.0, 4.0, 5.0]])

## 3. GDP と生活水準

$$ \text{GDP}(t) = \text{GDP}_0 \cdot \frac{\text{人口}(t)}{\text{人口}_0} \cdot H^{0.25} A^{0.30} I^{0.25} S^{0.15} $$

- 投資額そのものは GDP に足しません（**二重計上しない**）。GDP は状態変数だけから決まります。
- $\log$ GDP の変化は「人口の寄与 + Σ 弾力性 × log(状態の比)」に分解できます（どの状態変数が GDP を動かしたか）。

$$ \text{生活水準} = 0.4 \times \text{1 人あたり GDP 指数} + 0.35 \times \text{健康指数} + 0.25 \times \text{人的資本指数} $$

生活水準は**教育用の人工的な指標**で、現実の幸福度や社会的厚生を表すものではありません。

## 4. 20 年間のシミュレーション（均等配分）

In [ ]:
recs = sim.simulate(al.equal_allocation(), PARAMS)
table([recs[t] for t in (0, 1, 2, 3, 4, 5, 10, 15, 20)],
      [("year", "year", "{}"), ("gdp", "GDP", "{:.1f}"), ("gdp_growth_rate", "growth", "{:.2%}"),
       ("human_capital", "human cap.", "{:.3f}"), ("technology", "technology", "{:.3f}"),
       ("infrastructure", "infra.", "{:.3f}"), ("health", "health", "{:.3f}"),
       ("living_standard", "living std", "{:.3f}")])
d = em.gdp_decomposition({s: recs[0][s] for s in em.STATES}, {s: recs[-1][s] for s in em.STATES},
                         recs[0]["population"], recs[-1]["population"], PARAMS)
print("20 年間の log GDP の変化の内訳:", {k: round(v, 3) for k, v in d.items()})

In [ ]:
#@title 状態変数の時系列
show(gc.plot_state_panels(recs, "Equal allocation, 20 years (model)"), "01_state_panels")

## 5. 時系列のグラフ（シナリオ別）

GDP・人的資本・技術・生活水準を、5 つの配分シナリオで比べます（詳しくは Notebook 02）。

In [ ]:
#@title GDP・人的資本・技術・生活水準の時系列
runs = sim.run_scenarios(SCENARIOS, PARAMS)
show(gc.plot_series(runs, "gdp", "GDP over time (model)", "GDP (fictional units)", labels=LABELS), "01_gdp")
show(gc.plot_series(runs, "human_capital", "Human capital over time (model)", labels=LABELS), "01_human_capital")
show(gc.plot_series(runs, "technology", "Technology over time (model)", labels=LABELS), "01_technology")
show(gc.plot_series(runs, "living_standard", "Living standard over time (model)", labels=LABELS),
     "01_living_standard")

## 6. 学生向け TODO：パラメータを変えて成長経路を比べる

In [ ]:
# TODO 1：投資効率を変更する / TODO 2：減耗率を変更する / TODO 3：投資効果の遅延を変更する
variants = {
    "baseline": PARAMS,
    "R&D efficiency x2": em.with_changes(PARAMS, rd_efficiency=PARAMS["rd_efficiency"] * 2),
    "infra depreciation 0.10": em.with_changes(PARAMS, infrastructure_depreciation=0.10),
    "no delay": em.with_changes(PARAMS, education_delay=0, rd_delay=0),
}
runs2 = {name: sim.simulate(al.equal_allocation(), p) for name, p in variants.items()}
# TODO 4：成長経路を比較する
show(gc.plot_series(runs2, "gdp", "Equal allocation: GDP under different assumptions (model)", "GDP"), "01_todo")

## 考察（チームで記入）

- **TODO**：どの状態変数が GDP の成長に最も寄与したか。それはモデルのどの仮定から来ているか
- **TODO**：減耗率や遅延を変えると、成長経路のどこが変わったか
- **TODO**：このモデルで表せない経済の仕組みは何か（モデルの限界）

## 参考文献

- 柴田望洋『新・明解 Pythonで学ぶアルゴリズムとデータ構造（第2版）』SBクリエイティブ（第2章 最大値、第3章 線形探索、第4章 キュー、第6章 単純挿入ソート）
- G1 リポジトリの `README.md`（モデルの仮定・計算量・モデルの限界）
- データ：`data/baseline_economy.json`、`data/investment_scenarios.json`（この教材のための**架空のデータ**）